# Phase 4 - Notebook 04: pixelSplat & Implicit Geometry

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase4/04_pixelsplat_implicit_geometry.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand pixelSplat's approach to feed-forward 3DGS without explicit Cost Volume
2. Implement a simplified epipolar cross-attention mechanism
3. Understand implicit vs explicit geometry learning
4. Compare MVSplat and pixelSplat architectures in detail
5. Analyze the trade-offs between the two approaches

**Estimated Time**: 75 minutes

**Prerequisites**: Notebook 03 (MVSplat Architecture)

---

In [ ]:
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")

## 1. pixelSplat vs MVSplat: Two Philosophies

### MVSplat (Notebook 03): Explicit Geometry
```
Features → Cost Volume → 3D CNN → Depth → Gaussians
             (explicit)   (regularize)   (from geometry)
```

### pixelSplat: Implicit Geometry
```
Features → Cross-Attention → Directly predict Gaussians
          (learn geometry)    (no explicit depth volume)
```

| Aspect | MVSplat | pixelSplat |
|--------|---------|------------|
| Geometry | Explicit (Cost Volume) | Implicit (learned) |
| Cross-view | Warp + compare | Attention |
| Backbone | Lightweight U-Net | Heavier CNN (EfficientNet) |
| Depth | From soft argmin | Predicted by decoder |
| Speed | Faster (~22 FPS) | Slower (~10 FPS) |
| Quality | Competitive | Slightly higher on some benchmarks |
| Memory | Cost Volume is large | Attention can be expensive |
| Paper | ECCV 2024 | CVPR 2024 |

In [ ]:
# Architecture comparison visualization

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

def draw_box(ax, x, y, w, h, text, color='lightblue', fontsize=9, subtext=None):
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='black', lw=1.5)
    ax.add_patch(box)
    dy = 0.15 if subtext else 0
    ax.text(x + w/2, y + h/2 + dy, text,
            ha='center', va='center', fontsize=fontsize, fontweight='bold')
    if subtext:
        ax.text(x + w/2, y + h/2 - 0.25, subtext,
                ha='center', va='center', fontsize=7, style='italic', color='#444')

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

# Left: MVSplat
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 12); ax.axis('off')
ax.set_title('MVSplat (Explicit Geometry)', fontsize=13, fontweight='bold')

draw_box(ax, 1.5, 10.5, 3, 0.8, 'Images', '#FFE0B2')
draw_box(ax, 5.5, 10.5, 3, 0.8, 'Camera Poses', '#E0E0E0')
draw_arrow(ax, 3, 10.5, 3, 9.8)
draw_box(ax, 1, 8.8, 4, 0.8, 'U-Net Encoder', '#BBDEFB', subtext='Lightweight, shared')
draw_arrow(ax, 3, 8.8, 3, 8.2)
draw_box(ax, 0.5, 6.8, 5, 1.0, 'Plane Sweeping Cost Volume', '#E1BEE7',
         subtext='[B, C, D, H, W] - Explicit depth!')
draw_arrow(ax, 3, 6.8, 3, 6.2)
draw_arrow(ax, 7, 10.5, 5, 7.3)
draw_box(ax, 1, 5.0, 4, 0.8, '3D CNN', '#F8BBD0', subtext='Regularize cost volume')
draw_arrow(ax, 3, 5.0, 3, 4.4)
draw_box(ax, 0.5, 3.2, 5, 0.8, 'Prediction Heads', '#B3E5FC', subtext='Depth + Scale + Rot + Opacity')
draw_arrow(ax, 3, 3.2, 3, 2.6)
draw_box(ax, 1, 1.5, 4, 0.8, 'Gaussians', '#A5D6A7', subtext='Back-project depth')

# Right: pixelSplat
ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 12); ax.axis('off')
ax.set_title('pixelSplat (Implicit Geometry)', fontsize=13, fontweight='bold')

draw_box(ax, 1.5, 10.5, 3, 0.8, 'Images', '#FFE0B2')
draw_box(ax, 5.5, 10.5, 3, 0.8, 'Camera Poses', '#E0E0E0')
draw_arrow(ax, 3, 10.5, 3, 9.8)
draw_box(ax, 0.5, 8.8, 5, 0.8, 'CNN Backbone', '#BBDEFB', subtext='EfficientNet/ResNet, heavier')
draw_arrow(ax, 3, 8.8, 3, 8.2)
draw_box(ax, 0.5, 6.8, 5, 1.0, 'Epipolar Cross-Attention', '#FFF9C4',
         subtext='Implicit geometry learning!')
draw_arrow(ax, 3, 6.8, 3, 6.2)
draw_arrow(ax, 7, 10.5, 5, 7.3)
draw_box(ax, 1, 5.0, 4, 0.8, 'Self-Attention', '#FFF9C4', subtext='Within-view reasoning')
draw_arrow(ax, 3, 5.0, 3, 4.4)
draw_box(ax, 0.5, 3.2, 5, 0.8, 'Gaussian Decoder', '#B3E5FC',
         subtext='Direct prediction (all params)')
draw_arrow(ax, 3, 3.2, 3, 2.6)
draw_box(ax, 1, 1.5, 4, 0.8, 'Gaussians', '#A5D6A7', subtext='Including depth')

plt.tight_layout()
plt.show()

print("Key difference: pixelSplat replaces the explicit Cost Volume")
print("with learned cross-view attention for geometry reasoning.")

## 2. CNN Backbone for pixelSplat

pixelSplat uses a **heavier backbone** than MVSplat because the geometry reasoning happens in the attention layers rather than a cost volume, so the features themselves must be more expressive.

### Common choices:
- **EfficientNet** (used in original pixelSplat)
- **ResNet** with Feature Pyramid Network (FPN)

Here we use a simple ResNet-style backbone for illustration.

In [ ]:
class ResidualBlock(nn.Module):
    """Basic residual block."""
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.conv(x))


class SimpleResNetBackbone(nn.Module):
    """
    Simplified ResNet-style backbone for pixelSplat.
    
    Heavier than MVSplat's U-Net because features need to carry
    more information for the cross-attention to work.
    """

    def __init__(self, in_channels=3, feature_dim=128):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.layer1 = nn.Sequential(
            ResidualBlock(64),
            ResidualBlock(64),
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            ResidualBlock(128),
        )
        # Upsample back to input resolution
        self.upsample = nn.Sequential(
            nn.ConvTranspose2d(128, feature_dim, 4, stride=4, padding=0),
            nn.BatchNorm2d(feature_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        """Input: [B, 3, H, W], Output: [B, feature_dim, H, W]."""
        x = self.stem(x)       # [B, 64, H/2, W/2]
        x = self.layer1(x)     # [B, 64, H/2, W/2]
        x = self.layer2(x)     # [B, 128, H/4, W/4]
        x = self.upsample(x)   # [B, feature_dim, H, W]
        return x


# Compare backbone sizes
backbone = SimpleResNetBackbone(feature_dim=128)
dummy = torch.randn(1, 3, 64, 64)

with torch.no_grad():
    out = backbone(dummy)

print(f"pixelSplat backbone:")
print(f"  Input:  {list(dummy.shape)}")
print(f"  Output: {list(out.shape)}")
print(f"  Params: {sum(p.numel() for p in backbone.parameters()):,}")
print(f"\nFor comparison, MVSplat's U-Net has fewer parameters")
print(f"because the Cost Volume handles geometric reasoning.")

## 3. Epipolar Geometry Review

### 3.1 Why Epipolar?

For a point at pixel $(u, v)$ in view 1, its corresponding point in view 2 **must lie on the epipolar line**. This constraint reduces the search space from 2D to 1D:

```
View 1                  View 2
┌─────────┐            ┌─────────┐
│    *     │            │ ─────── │  ← Epipolar line
│  (u,v)  │            │   *     │    (1D search)
└─────────┘            └─────────┘

Without epipolar: search entire 2D image
With epipolar:    search along 1D line only
```

### 3.2 Epipolar Line Computation

The epipolar line in view 2 for a point $\mathbf{p}$ in view 1 is:
$$\mathbf{l}_2 = F \cdot \mathbf{p}_1$$

where $F$ is the **Fundamental Matrix** relating the two views.

In [ ]:
def compute_fundamental_matrix(K1, K2, R, t):
    """
    Compute the fundamental matrix F from camera parameters.
    F = K2^{-T} @ [t]_x @ R @ K1^{-1}
    """
    # Skew-symmetric matrix for cross product
    tx = torch.tensor([
        [0, -t[2], t[1]],
        [t[2], 0, -t[0]],
        [-t[1], t[0], 0]
    ], dtype=torch.float32)

    E = tx @ R  # Essential matrix
    F = torch.inverse(K2).T @ E @ torch.inverse(K1)  # Fundamental matrix
    return F


def compute_epipolar_line(F, point):
    """Compute epipolar line l = F @ p (in homogeneous coordinates)."""
    p_homo = torch.tensor([point[0], point[1], 1.0])
    line = F @ p_homo
    # Normalize: ax + by + c = 0
    line = line / torch.sqrt(line[0]**2 + line[1]**2)
    return line


# Setup cameras
H, W = 64, 64
K = torch.tensor([[50, 0, W/2], [0, 50, H/2], [0, 0, 1]], dtype=torch.float32)
angle = np.radians(5.0)
R = torch.tensor([[np.cos(angle), 0, np.sin(angle)],
                   [0, 1, 0],
                   [-np.sin(angle), 0, np.cos(angle)]], dtype=torch.float32)
t = torch.tensor([0.5, 0.0, 0.0])

F_matrix = compute_fundamental_matrix(K, K, R, t)

# Visualize epipolar lines for several points
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

test_points = [(20, 20), (40, 30), (15, 50), (50, 10), (32, 32)]
colors = plt.cm.Set1(np.linspace(0, 1, len(test_points)))

# View 1: show points
ax = axes[0]
ax.set_title('View 1: Source Points', fontsize=12, fontweight='bold')
ax.set_xlim(0, W); ax.set_ylim(H, 0)
ax.set_aspect('equal')
for (u, v), color in zip(test_points, colors):
    ax.scatter([u], [v], c=[color], s=100, zorder=5, edgecolors='black')
    ax.text(u + 2, v - 2, f'({u},{v})', fontsize=8)
ax.set_xlabel('u'); ax.set_ylabel('v')
ax.grid(True, alpha=0.2)

# View 2: show epipolar lines
ax = axes[1]
ax.set_title('View 2: Epipolar Lines', fontsize=12, fontweight='bold')
ax.set_xlim(0, W); ax.set_ylim(H, 0)
ax.set_aspect('equal')

for (u, v), color in zip(test_points, colors):
    line = compute_epipolar_line(F_matrix, (u, v))
    a, b, c = line.numpy()
    # Draw line: ax + by + c = 0 → y = -(ax + c) / b
    x_range = np.linspace(0, W, 100)
    if abs(b) > 1e-6:
        y_range = -(a * x_range + c) / b
        mask = (y_range >= 0) & (y_range <= H)
        ax.plot(x_range[mask], y_range[mask], color=color, lw=2, alpha=0.8,
                label=f'({u},{v})')

ax.legend(fontsize=8, title='Source point')
ax.set_xlabel('u'); ax.set_ylabel('v')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print("Each point in View 1 constrains the search to a LINE in View 2.")
print("pixelSplat's cross-attention operates ALONG these epipolar lines.")

## 4. Epipolar Cross-Attention

### 4.1 Core Idea

Instead of building a Cost Volume (warp + compare), pixelSplat uses **attention** to let each pixel "look at" relevant features in the other view along its epipolar line:

```
Standard Cross-Attention: Q from view1, K/V from ALL of view2
                          → O(H*W * H*W) = very expensive

Epipolar Cross-Attention: Q from view1, K/V from EPIPOLAR LINE in view2
                          → O(H*W * S) where S = samples on line
                          → much more efficient
```

### 4.2 Steps

1. For each pixel in view 1, compute its epipolar line in view 2
2. Sample S points along the epipolar line
3. Extract features at these S points (bilinear interpolation)
4. Compute attention: query (view 1 pixel) attends to keys/values (S epipolar samples)
5. Output: geometry-aware feature for each pixel

In [ ]:
class EpipolarSampler(nn.Module):
    """
    Sample points along epipolar lines.
    
    For each pixel in the reference view, computes the epipolar line
    in the source view and samples S evenly-spaced points along it.
    """

    def __init__(self, num_samples=32):
        super().__init__()
        self.num_samples = num_samples

    def forward(self, K_ref, K_src, T_src_ref, H, W):
        """
        Compute epipolar sample locations.
        
        Args:
            K_ref: [B, 3, 3] reference intrinsics
            K_src: [B, 3, 3] source intrinsics
            T_src_ref: [B, 4, 4] relative pose
            H, W: image dimensions
        
        Returns:
            sample_coords: [B, H*W, S, 2] normalized coords in source view
        """
        B = K_ref.shape[0]
        S = self.num_samples
        device = K_ref.device

        R = T_src_ref[:, :3, :3]
        t = T_src_ref[:, :3, 3]

        # Create pixel grid
        u = torch.arange(W, device=device, dtype=torch.float32)
        v = torch.arange(H, device=device, dtype=torch.float32)
        vv, uu = torch.meshgrid(v, u, indexing='ij')
        ones = torch.ones_like(uu)
        pixels = torch.stack([uu, vv, ones], dim=-1).reshape(-1, 3)  # [H*W, 3]
        pixels = pixels.unsqueeze(0).expand(B, -1, -1)  # [B, H*W, 3]

        # Compute rays in reference frame
        K_ref_inv = torch.inverse(K_ref)  # [B, 3, 3]
        rays = torch.bmm(pixels, K_ref_inv.transpose(1, 2))  # [B, H*W, 3]

        # Sample at different depths along each ray
        # Use log-uniform depths (similar to cost volume depth planes)
        depths = torch.exp(torch.linspace(
            np.log(1.0), np.log(20.0), S, device=device
        ))  # [S]

        # For each pixel, project at each depth to source view
        all_coords = []
        for d_idx in range(S):
            depth = depths[d_idx]
            # 3D points: ray * depth
            pts_3d = rays * depth  # [B, H*W, 3]

            # Transform to source frame
            pts_src = torch.bmm(pts_3d, R.transpose(1, 2)) + t.unsqueeze(1)

            # Project to source image
            pts_proj = torch.bmm(pts_src, K_src.transpose(1, 2))  # [B, H*W, 3]
            coords = pts_proj[:, :, :2] / (pts_proj[:, :, 2:3] + 1e-8)

            # Normalize to [-1, 1]
            coords_norm = coords.clone()
            coords_norm[:, :, 0] = 2.0 * coords[:, :, 0] / (W - 1) - 1.0
            coords_norm[:, :, 1] = 2.0 * coords[:, :, 1] / (H - 1) - 1.0

            all_coords.append(coords_norm)

        sample_coords = torch.stack(all_coords, dim=2)  # [B, H*W, S, 2]
        return sample_coords


# Test the sampler
sampler = EpipolarSampler(num_samples=16)

K_batch = K.unsqueeze(0)
T_batch = torch.eye(4).unsqueeze(0)
T_batch[0, :3, :3] = R
T_batch[0, :3, 3] = t

coords = sampler(K_batch, K_batch, T_batch, H=32, W=32)
print(f"Epipolar sample coordinates: {list(coords.shape)}")
print(f"  B={coords.shape[0]}, H*W={coords.shape[1]}, S={coords.shape[2]}")
print(f"  For each of {coords.shape[1]} pixels, we sample {coords.shape[2]} points")
print(f"  along the epipolar line in the source view.")

In [ ]:
# Visualize epipolar samples for a few pixels

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
H_test, W_test = 32, 32

test_pixels = [(8, 8), (16, 16), (24, 8), (8, 24)]
colors_list = ['red', 'blue', 'green', 'orange']

# View 1: mark source pixels
ax = axes[0]
ax.set_title('View 1: Query Pixels', fontsize=12, fontweight='bold')
ax.set_xlim(0, W_test); ax.set_ylim(W_test, 0)
for (u, v), c in zip(test_pixels, colors_list):
    ax.scatter([u], [v], c=c, s=150, zorder=5, edgecolors='black', marker='*')
    ax.text(u + 1, v - 1, f'({u},{v})', fontsize=8, color=c)
ax.grid(True, alpha=0.2); ax.set_aspect('equal')
ax.set_xlabel('u'); ax.set_ylabel('v')

# View 2: show epipolar samples
ax = axes[1]
ax.set_title('View 2: Epipolar Samples (K/V locations)', fontsize=12, fontweight='bold')
ax.set_xlim(-1.2, 1.2); ax.set_ylim(1.2, -1.2)

for (u, v), c in zip(test_pixels, colors_list):
    pixel_idx = v * W_test + u
    samples = coords[0, pixel_idx].numpy()  # [S, 2]
    ax.scatter(samples[:, 0], samples[:, 1], c=c, s=30, alpha=0.8,
              edgecolors='black', lw=0.5, label=f'({u},{v})')
    ax.plot(samples[:, 0], samples[:, 1], c=c, alpha=0.3, lw=1)

ax.legend(fontsize=8, title='Source pixel')
ax.set_xlabel('Normalized u'); ax.set_ylabel('Normalized v')
ax.grid(True, alpha=0.2); ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print("Each query pixel samples S points along its epipolar line.")
print("The attention mechanism learns which sample corresponds to the true match.")

In [ ]:
class EpipolarCrossAttention(nn.Module):
    """
    Cross-attention along epipolar lines.
    
    For each pixel in view 1 (query), attend to features sampled
    along the epipolar line in view 2 (keys/values).
    
    This replaces the Cost Volume in MVSplat:
    - Cost Volume: explicitly warp + compare at D depths
    - Epipolar Attention: learn to match along the epipolar line
    """

    def __init__(self, feature_dim=128, num_heads=4, num_samples=32):
        super().__init__()
        self.feature_dim = feature_dim
        self.num_heads = num_heads
        self.num_samples = num_samples
        self.head_dim = feature_dim // num_heads

        # Query, Key, Value projections
        self.q_proj = nn.Linear(feature_dim, feature_dim)
        self.k_proj = nn.Linear(feature_dim, feature_dim)
        self.v_proj = nn.Linear(feature_dim, feature_dim)
        self.out_proj = nn.Linear(feature_dim, feature_dim)

        # Epipolar sampler
        self.sampler = EpipolarSampler(num_samples)

        self.scale = self.head_dim ** -0.5

    def forward(self, feat_ref, feat_src, K_ref, K_src, T_src_ref):
        """
        Epipolar cross-attention.
        
        Args:
            feat_ref: [B, C, H, W] reference features
            feat_src: [B, C, H, W] source features
            K_ref, K_src: [B, 3, 3] intrinsics
            T_src_ref: [B, 4, 4] relative pose
        
        Returns:
            out: [B, C, H, W] geometry-aware features
            attn_weights: [B, num_heads, H*W, S] attention maps
        """
        B, C, H, W = feat_ref.shape
        N = H * W
        S = self.num_samples

        # Step 1: Get epipolar sample coordinates
        sample_coords = self.sampler(
            K_ref, K_src, T_src_ref, H, W
        )  # [B, N, S, 2]

        # Step 2: Sample source features at epipolar locations
        # Reshape for grid_sample: [B, N*S, 1, 2] → sample from [B, C, H, W]
        grid = sample_coords.reshape(B, N * S, 1, 2)  # [B, N*S, 1, 2]
        sampled_feats = F.grid_sample(
            feat_src, grid, mode='bilinear',
            padding_mode='zeros', align_corners=True
        )  # [B, C, N*S, 1]
        sampled_feats = sampled_feats.squeeze(-1)  # [B, C, N*S]
        sampled_feats = sampled_feats.reshape(B, C, N, S)  # [B, C, N, S]
        sampled_feats = sampled_feats.permute(0, 2, 3, 1)  # [B, N, S, C]

        # Step 3: Prepare Q, K, V
        # Query: from reference features
        ref_flat = feat_ref.flatten(2).permute(0, 2, 1)  # [B, N, C]
        Q = self.q_proj(ref_flat)  # [B, N, C]

        # Key, Value: from sampled source features
        K_attn = self.k_proj(sampled_feats.reshape(B * N, S, C))
        K_attn = K_attn.reshape(B, N, S, C)
        V = self.v_proj(sampled_feats.reshape(B * N, S, C))
        V = V.reshape(B, N, S, C)

        # Step 4: Multi-head attention
        Q = Q.reshape(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        K_attn = K_attn.reshape(B, N, S, self.num_heads, self.head_dim).permute(0, 3, 1, 2, 4)
        V = V.reshape(B, N, S, self.num_heads, self.head_dim).permute(0, 3, 1, 2, 4)

        # Q: [B, heads, N, head_dim], K: [B, heads, N, S, head_dim]
        attn = torch.einsum('bhnd,bhnsd->bhns', Q, K_attn) * self.scale
        attn_weights = F.softmax(attn, dim=-1)  # [B, heads, N, S]

        # Weighted sum of values
        out = torch.einsum('bhns,bhnsd->bhnd', attn_weights, V)
        out = out.permute(0, 2, 1, 3).reshape(B, N, C)  # [B, N, C]
        out = self.out_proj(out)  # [B, N, C]

        # Reshape back to spatial
        out = out.permute(0, 2, 1).reshape(B, C, H, W)  # [B, C, H, W]

        # Add residual
        out = out + feat_ref

        return out, attn_weights


# Test the module
torch.manual_seed(42)
epi_attn = EpipolarCrossAttention(feature_dim=128, num_heads=4, num_samples=16)

B, C, H, W = 1, 128, 16, 16
feat_r = torch.randn(B, C, H, W)
feat_s = torch.randn(B, C, H, W)

with torch.no_grad():
    out, attn_w = epi_attn(feat_r, feat_s, K_batch, K_batch, T_batch)

print(f"Input features:     {list(feat_r.shape)}")
print(f"Output features:    {list(out.shape)}")
print(f"Attention weights:  {list(attn_w.shape)}  (B, heads, H*W, S)")
print(f"Parameters: {sum(p.numel() for p in epi_attn.parameters()):,}")

In [ ]:
# Visualize attention maps for selected pixels

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

test_pixel_indices = [0, 64, 128, 200]
head_labels = ['Head 0', 'Head 1', 'Head 2', 'Head 3']

for col, pix_idx in enumerate(test_pixel_indices):
    py, px = pix_idx // W, pix_idx % W

    # Top row: attention weights across heads for this pixel
    ax = axes[0, col]
    for h in range(4):
        weights = attn_w[0, h, pix_idx].detach().numpy()
        ax.plot(weights, label=f'Head {h}', alpha=0.8)
    ax.set_title(f'Pixel ({px},{py})', fontsize=10, fontweight='bold')
    ax.set_xlabel('Epipolar sample idx')
    ax.set_ylabel('Attention weight')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    # Bottom row: where attention focuses
    ax = axes[1, col]
    mean_attn = attn_w[0, :, pix_idx].mean(dim=0).detach().numpy()  # [S]
    ax.bar(range(len(mean_attn)), mean_attn, color='steelblue', alpha=0.7)
    best_idx = np.argmax(mean_attn)
    ax.bar(best_idx, mean_attn[best_idx], color='red', alpha=0.9)
    ax.set_xlabel('Epipolar sample idx')
    ax.set_ylabel('Mean attention')
    ax.set_title(f'Best match @ sample {best_idx}', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Epipolar Cross-Attention: Each pixel learns to attend to its best match',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("The attention peak indicates where the network believes")
print("the true correspondence lies along the epipolar line.")
print("This implicitly encodes depth information.")

## 5. Complete pixelSplat Architecture

### 5.1 Simplified Implementation

In [ ]:
from src.feedforward.gaussian_predictor import GaussianPredictionHeads
from src.feedforward.pixel_aligned import PixelAlignedGaussians


class SimplifiedPixelSplat(nn.Module):
    """
    Simplified pixelSplat architecture for educational purposes.
    
    Pipeline: images → backbone → cross-attention → self-attention → predict Gaussians
    
    Key difference from MVSplat: NO explicit Cost Volume.
    Geometry is learned implicitly through attention.
    """

    def __init__(self, feature_dim=128, num_heads=4, num_epi_samples=16):
        super().__init__()

        # 1. Backbone (heavier than MVSplat's U-Net)
        self.backbone = SimpleResNetBackbone(feature_dim=feature_dim)

        # 2. Epipolar cross-attention (replaces Cost Volume)
        self.cross_attention = EpipolarCrossAttention(
            feature_dim=feature_dim,
            num_heads=num_heads,
            num_samples=num_epi_samples,
        )

        # 3. Self-attention for within-view reasoning
        self.self_attention = nn.MultiheadAttention(
            embed_dim=feature_dim,
            num_heads=num_heads,
            batch_first=True,
        )
        self.sa_norm = nn.LayerNorm(feature_dim)

        # 4. Gaussian prediction heads
        self.gaussian_heads = GaussianPredictionHeads(
            in_channels=feature_dim,
            hidden_channels=64,
            depth_mode='regression',
            covariance_mode='3d',
        )

    def forward(self, images, K, poses, T_relative):
        """
        pixelSplat forward pass.
        
        Args:
            images: list of [B, 3, H, W] (2 views)
            K: [B, 3, 3] intrinsics
            poses: list of [B, 4, 4] camera poses
            T_relative: [B, 4, 4] ref-to-source transform
        
        Returns:
            PixelAlignedGaussians, predictions dict, shape trace
        """
        shapes = {}

        # Step 1: Extract features with backbone
        feat_ref = self.backbone(images[0])
        feat_src = self.backbone(images[1])
        shapes['backbone_features'] = list(feat_ref.shape)

        # Step 2: Epipolar cross-attention (replaces Cost Volume!)
        feat_cross, attn_weights = self.cross_attention(
            feat_ref, feat_src, K, K, T_relative
        )
        shapes['cross_attention'] = list(feat_cross.shape)
        shapes['attention_weights'] = list(attn_weights.shape)

        # Step 3: Self-attention
        B, C, H, W = feat_cross.shape
        feat_flat = feat_cross.flatten(2).permute(0, 2, 1)  # [B, H*W, C]
        feat_sa, _ = self.self_attention(feat_flat, feat_flat, feat_flat)
        feat_sa = self.sa_norm(feat_sa + feat_flat)  # residual + norm
        feat_final = feat_sa.permute(0, 2, 1).reshape(B, C, H, W)
        shapes['self_attention'] = list(feat_final.shape)

        # Step 4: Predict Gaussians
        predictions = self.gaussian_heads(feat_final)
        for key, val in predictions.items():
            shapes[f'pred_{key}'] = list(val.shape)

        # Step 5: Create pixel-aligned Gaussians
        gaussians = PixelAlignedGaussians.from_depth_and_features(
            depth=predictions['depth'],
            features={
                'scales': predictions['scales'],
                'rotations': predictions['rotations'],
                'opacities': predictions['opacities'],
            },
            K=K,
            pose=poses[0],
            image_colors=images[0],
        )
        shapes['gaussians'] = list(gaussians.positions.shape)

        return gaussians, predictions, shapes


# Create and test
pixelsplat = SimplifiedPixelSplat(feature_dim=128, num_heads=4, num_epi_samples=16)
pixelsplat.eval()

B, H, W = 1, 32, 32
imgs = [torch.randn(B, 3, H, W), torch.randn(B, 3, H, W)]
K_b = torch.tensor([[30, 0, W/2], [0, 30, H/2], [0, 0, 1]], dtype=torch.float32).unsqueeze(0)
pose0 = torch.eye(4).unsqueeze(0)

with torch.no_grad():
    gs, preds, shapes = pixelsplat(imgs, K_b, [pose0, T_batch], T_batch)

print("=" * 60)
print("pixelSplat Forward Pass - Tensor Shape Trace")
print("=" * 60)
for stage, shape in shapes.items():
    print(f"  {stage:25s}: {shape}")

total_params = sum(p.numel() for p in pixelsplat.parameters())
print(f"\nTotal parameters: {total_params:,}")

## 6. Explicit vs Implicit Geometry: Deep Comparison

### 6.1 What Does "Implicit Geometry" Mean?

**Explicit** (MVSplat): The Cost Volume directly stores matching costs at each depth. The depth is *read out* from this explicit structure.

**Implicit** (pixelSplat): The attention layers learn to reason about depth through feature matching, but depth is never stored as an explicit volume. Depth emerges from the learned features.

### 6.2 Computational Comparison

In [ ]:
# Computational comparison

def compute_flops_estimate(H, W, C, D, S, num_heads):
    """Rough FLOPs comparison (order of magnitude)."""
    N = H * W

    # MVSplat Cost Volume: warp at D depths + compare
    cv_warp_flops = D * N * C * 2  # warp + compare at each depth
    cv_3dcnn_flops = D * N * C * C * 9  # 3x3x3 conv
    mvsplat_flops = cv_warp_flops + cv_3dcnn_flops

    # pixelSplat: Q/K/V projection + attention
    qkv_flops = 3 * N * C * C  # Q, K, V projections
    attn_flops = N * S * C  # attention computation
    sa_flops = N * N * C  # self-attention (expensive for large N!)
    pixelsplat_flops = qkv_flops + attn_flops + sa_flops

    return mvsplat_flops, pixelsplat_flops


resolutions = [(32, 32), (64, 64), (128, 128), (256, 256)]
C, D, S = 64, 32, 32

print(f"{'Resolution':>12s} | {'MVSplat FLOPs':>15s} | {'pixelSplat FLOPs':>18s} | Ratio")
print("-" * 75)

mv_flops_list = []
ps_flops_list = []

for h, w in resolutions:
    mv, ps = compute_flops_estimate(h, w, C, D, S, 4)
    ratio = ps / mv
    mv_flops_list.append(mv)
    ps_flops_list.append(ps)
    print(f"{h:>4d}x{w:<4d}    | {mv:>12,.0f}    | {ps:>15,.0f}    | {ratio:.2f}x")

# Plot scaling behavior
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
res_labels = [f'{h}x{w}' for h, w in resolutions]
x = range(len(resolutions))

ax.bar([i - 0.2 for i in x], [f/1e6 for f in mv_flops_list], width=0.35,
       label='MVSplat', color='#2196F3', alpha=0.8)
ax.bar([i + 0.2 for i in x], [f/1e6 for f in ps_flops_list], width=0.35,
       label='pixelSplat', color='#FF9800', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(res_labels)
ax.set_xlabel('Image Resolution')
ax.set_ylabel('Estimated FLOPs (M)')
ax.set_title('Computational Cost Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nKey insight: pixelSplat's self-attention scales as O(N^2),")
print("making it more expensive at higher resolutions.")
print("MVSplat's Cost Volume scales as O(N*D), which is more manageable.")

## 7. Strengths and Weaknesses

### pixelSplat Strengths

1. **Flexible geometry**: Attention can handle non-planar surfaces better
2. **End-to-end**: No hand-crafted depth hypotheses needed
3. **Richer features**: Attention captures long-range dependencies
4. **Robustness**: Can handle larger viewpoint changes

### pixelSplat Weaknesses

1. **Slower**: Self-attention is O(N^2)
2. **More parameters**: Heavier backbone needed
3. **Less interpretable**: Hard to visualize what geometry was learned
4. **More data hungry**: Implicit learning needs more training data

### When to Use Which?

| Scenario | Recommended |
|----------|------------|
| Real-time applications | MVSplat (faster) |
| Highest quality | pixelSplat (slightly better on some metrics) |
| Limited training data | MVSplat (explicit geometry is a strong prior) |
| Wide baselines | pixelSplat (more flexible matching) |
| Resource constrained | MVSplat (lighter model) |
| Research / extension | Both are good starting points |

In [ ]:
# Summary comparison

summary = """
=====================================================================
  Notebook 04 Summary: pixelSplat & Implicit Geometry
=====================================================================

1. TWO APPROACHES TO FEED-FORWARD 3DGS
   - MVSplat: Cost Volume (explicit geometry)
   - pixelSplat: Cross-Attention (implicit geometry)

2. EPIPOLAR CROSS-ATTENTION
   - Query: pixel features from reference view
   - Key/Value: features sampled along epipolar line
   - Learns to match without explicit depth hypotheses
   - Replaces the Cost Volume + 3D CNN

3. ARCHITECTURE DIFFERENCES
   - pixelSplat uses heavier backbone (more parameters)
   - pixelSplat has no explicit depth volume
   - pixelSplat includes self-attention (O(N^2))

4. TRADE-OFFS
   ┌─────────────┬──────────────┬──────────────┐
   |             | MVSplat      | pixelSplat   |
   ├─────────────┼──────────────┼──────────────┤
   | Speed       | Faster       | Slower       |
   | Quality     | Competitive  | Slightly +   |
   | Geometry    | Explicit     | Implicit     |
   | Parameters  | Lighter      | Heavier      |
   | Scalability | Better       | O(N^2) cost  |
   └─────────────┴──────────────┴──────────────┘

5. IMPLICIT GEOMETRY LEARNING
   - Depth is not stored in a volume
   - Attention weights implicitly encode matches
   - Network learns geometry from photometric loss

=====================================================================
"""
print(summary)

## What's Next?

**[05_training_loss_design.ipynb](./05_training_loss_design.ipynb)** - Training strategies and loss functions for feed-forward Gaussian methods, covering dataset pipelines, photometric losses, and evaluation metrics.

---

## References

1. pixelSplat: https://arxiv.org/abs/2312.12337
2. MVSplat: https://arxiv.org/abs/2403.14627
3. Epipolar Transformers: https://arxiv.org/abs/2005.04551